# Use Case 7 — BFSI Industry Case Studies

Three BFSI agent case studies built on the AgentTune agentic spine: fraud triage, KYC-onboarding distillation, and compliance graph memory.

## 1. Fraud triage, end to end

In [ ]:
"""
BFSI case study — Fraud-triage agent, end to end.
=================================================

A card-fraud triage agent that inspects a flagged transaction with tools, decides
APPROVE / REVIEW / BLOCK, and then — the point of the spine — its trajectories feed both
evaluation and failure-recovery. A stuck agent that loops on one check is caught by the real
FailureDetector and turned into corrective training data. GPU-free.
"""
from agenttune.agentic import (
    Project, DictToolHarness, ReActStrategy, run_episode, SelfHealLoop,
    Event, EventKind, EventLog, agentic_metrics,
)
from agenttune.decide.closed_loop.contracts import ClassifiedFailure, TrainingExample

In [2]:
# --- the fraud-analyst tools (deterministic stand-ins for real risk services) ---
def check_velocity(account=""):        # txns per hour on this account
    return {"account": account, "txn_per_hour": 9, "threshold": 5}

In [3]:
def check_geo(account=""):             # geo mismatch vs home region
    return {"account": account, "home": "US-TX", "txn_geo": "RO-B", "mismatch": True}

In [4]:
def sanctions_screen(name=""):
    return {"name": name, "ofac_hit": False}

In [5]:
TOOLS = {"check_velocity": check_velocity, "check_geo": check_geo, "sanctions_screen": sanctions_screen}

In [6]:
def analyst_policy(state):
    """A ReAct fraud analyst: velocity → geo → decision. Model-free, canned."""
    if state.step == 0:
        return {"name": "check_velocity", "arguments": {"account": "acct-4417"},
                "thought": "unusual spend — check velocity first"}
    if state.step == 1:
        return {"name": "check_geo", "arguments": {"account": "acct-4417"},
                "thought": "velocity is high; check for geo mismatch"}
    return {"name": "finish", "arguments": {"answer": "BLOCK"},
            "thought": "high velocity + geo mismatch → block and open a case"}

In [7]:
def main():
    proj = Project(strategy=ReActStrategy(analyst_policy, max_steps=5),
                   harness=DictToolHarness(TOOLS, max_steps=5))

    # 1) Triage a batch of flagged transactions (light-tier episodes).
    for txn in ("txn-1001 $4,200 electronics", "txn-1002 $980 gift cards"):
        log = proj.infer(f"triage flagged transaction: {txn}")
    print(f"[triage]   {len(proj.trajectories)} transactions triaged, "
          f"last decision reached in {len(log)} events")

    # 2) Evaluate the triage trajectories with the real programmatic metrics.
    per = [agentic_metrics(l) for l in proj.trajectories]
    mean = {k: round(sum(p[k] for p in per) / len(per), 2) for k in per[0]}
    print(f"[evaluate] mean trajectory metrics -> {mean}")

    # 3) A degraded agent slips out: it loops on check_velocity, never deciding.
    stuck = EventLog(tier="light")
    for _ in range(4):
        stuck.append(Event(EventKind.TOOL_CALL,
                           {"action": {"name": "check_velocity", "arguments": {"account": "acct-9"}}}))
        stuck.append(Event(EventKind.TOOL_RESULT, {"output": "txn_per_hour: 9"}))
    proj.add_trajectory(stuck)
    failures = proj.heal()
    print(f"[monitor]  FailureDetector -> {len(failures)} stuck agent(s): {[f.failure_type for f in failures]}")

    # 4) Turn the failure into corrective preference data (retrain-ready).
    classify = lambda fs: [ClassifiedFailure(failure=f, root_cause=f.failure_type, confidence=0.85,
                                             analysis="re-checks velocity without deciding") for f in fs]
    generate = lambda cfs: [TrainingExample(
        trajectory_id=c.failure.trajectory_id, original_failure_type=c.failure.failure_type,
        root_cause=c.root_cause,
        prompt=[{"role": "user", "content": "triage the flagged transaction"}],
        chosen=[{"role": "assistant", "content": "check geo, then decide BLOCK/REVIEW/APPROVE"}],
        rejected=[{"role": "assistant", "content": "check velocity again"}],
    ) for c in cfs]
    summary = SelfHealLoop(classify, generate).run(failures)
    print(f"[heal]     corrective dataset -> {summary['n_dataset_rows']} preference row(s) "
          f"{list(summary['dataset'][0].keys())}; ready to retrain the fraud agent")

In [8]:
if __name__ == "__main__":
    main()

[triage]   2 transactions triaged, last decision reached in 13 events


[evaluate] mean trajectory metrics -> {'tac': 0.0, 'ter': 1.0, 'arr': 0.0, 'scsr': 1.0, 'rad': 0.0, 'lcf': 0.0}
[monitor]  FailureDetector -> 1 stuck agent(s): ['loop_collapse']
[heal]     corrective dataset -> 1 preference row(s) ['prompt', 'chosen', 'rejected']; ready to retrain the fraud agent


## 2. KYC-onboarding agentic distillation

In [ ]:
"""
BFSI case study — Distil a KYC-triage agent into a small on-prem model.
======================================================================

Banks often cannot send customer PII to a large hosted model, and want inference cheap and
on-premises. Agentic distillation fits: run a strong teacher agent to produce KYC-triage
trajectories, then behavior-clone them into a small (<3B) student that can run in the bank's
own VPC. The teacher's *design* is compressed into the student — not its weights. GPU-free
here; the injected trainer is where the real fine-tune would run on the bank's GPU.
"""
from agenttune.agentic import Project
from agenttune.agentic.rollout_engines.demo_engine import DemoRolloutEngine

In [10]:
class OnPremTrainerFactory:
    """Stand-in for the bank's on-prem SFT trainer. Records what the spine hands it."""

    def __init__(self):
        self.captured = {}

    def __call__(self, *, model, train_dataset, **kwargs):
        self.captured = {"model": model, "n_rows": len(train_dataset)}

        class _Trainer:
            def train(_self):
                return {"status": "would-train-in-bank-vpc",
                        "student": self.captured["model"], "n_rows": self.captured["n_rows"]}
        return _Trainer()

In [11]:
def main():
    # 1) TEACHER — a capable KYC analyst agent triages onboarding cases (GPU-free engine).
    teacher = Project()
    cases = [
        "onboard: sole trader, cash-intensive business, PEP=no",
        "onboard: non-resident, complex ownership, source-of-funds unclear",
        "onboard: salaried applicant, clean sanctions screen",
    ]
    teacher.collect_rollout(DemoRolloutEngine(), cases, tools=[], max_steps=2)
    print(f"[teacher]  {len(teacher.trajectories)} KYC-triage trajectories collected")

    rows = teacher.sft_dataset()
    print(f"[teacher]  behavior-cloning dataset -> {len(rows)} SFT rows (schema {list(rows[0].keys())})")

    # 2) DISTILL — compress the analyst design into a small on-prem student.
    factory = OnPremTrainerFactory()
    result = teacher.distill("kyc-analyst-1.3B", trainer_factory=factory)
    print(f"[distill]  student='{factory.captured['model']}' fed {factory.captured['n_rows']} teacher rows")
    print(f"[distill]  result -> {result}")

In [12]:
if __name__ == "__main__":
    main()

[teacher]  3 KYC-triage trajectories collected
[teacher]  behavior-cloning dataset -> 3 SFT rows (schema ['messages', 'segment_weights', 'loss_mask'])
[distill]  student='kyc-analyst-1.3B' fed 3 teacher rows
[distill]  result -> {'status': 'would-train-in-bank-vpc', 'student': 'kyc-analyst-1.3B', 'n_rows': 3}


/workspace/agenttune/src/agenttune/agentic/project.py:140: UserWarning: No tools provided — running in single-turn mode.
  fn = create_rollout_fn(


## 3. Compliance graph memory

In [ ]:
"""
BFSI case study — Compliance QA over a regulation graph.
========================================================

Compliance knowledge is relational: a control implements a regulation, which cites a clause,
which supersedes an older one. A vector store recalls text that *looks* similar; a graph
recalls what is actually *connected*. This seeds a small regulation graph and shows an agent
recalling the controls linked to a regulation — the retrieval a compliance-QA agent needs.
GPU-free (no embedder, no model).
"""
from agenttune.agentic import GraphMemory, MemoryItem

In [14]:
def main():
    g = GraphMemory()

    # Nodes: two regulations, their controls, and an unrelated one.
    aml = g.write(MemoryItem(content="AML-4: ongoing transaction monitoring"))
    ctrl_velocity = g.write(MemoryItem(content="Control: velocity thresholds"))
    ctrl_sanctions = g.write(MemoryItem(content="Control: daily sanctions screening"))
    kyc = g.write(MemoryItem(content="KYC-1: customer due diligence"))
    ctrl_idv = g.write(MemoryItem(content="Control: identity verification"))
    unrelated = g.write(MemoryItem(content="Control: office fire drill"))  # present, unlinked

    # Edges: which control implements which regulation.
    g.link(aml, ctrl_velocity, "implemented_by")
    g.link(aml, ctrl_sanctions, "implemented_by")
    g.link(kyc, ctrl_idv, "implemented_by")

    # An auditor asks: which controls cover AML-4?
    recalled = [i.content for i in g.read("AML-4: ongoing transaction monitoring", k=5)]
    print("[query]  controls covering 'AML-4':")
    for c in recalled:
        print(f"           - {c}")
    print(f"[check]  KYC control surfaced? {'Control: identity verification' in recalled}  "
          f"(should be False — different regulation)")
    print(f"[check]  fire-drill surfaced? {'Control: office fire drill' in recalled}  "
          f"(should be False — unlinked)")

In [15]:
if __name__ == "__main__":
    main()

[query]  controls covering 'AML-4':
           - Control: velocity thresholds
           - Control: daily sanctions screening
[check]  KYC control surfaced? False  (should be False — different regulation)
[check]  fire-drill surfaced? False  (should be False — unlinked)
